# med-data-EDA · 阶段 B（全量外接盘）

PMC `oa_comm` 全量：**下载 → 解压 → slim jsonl → 策略抽样验证**。

| 项 | 说明 |
|---|---|
| 验证期分析 | 见 **`med-data-EDA-partA.ipynb`**（只读回顾，不再改） |
| 环境 | Open Folder → `02 数据处理/`，内核 **`med-rag-verify`** |
| 外接盘 | `/Volumes/Lexar/med-llm-rag-datasets`（`schedule.md`） |
| 定稿策略 | partA §6 + `outputs/tables/chunk_strategy_config.json` |

**运行顺序**：`【前置】` → **B0** → **B1** → … → **B6**（逐 cell，勿跳步）。

## 【前置】工程路径与对照表目录

定位 `02 数据处理/src`，并设置 `TABLES_DIR`（读取 partA 产出的验证期对照 CSV）。  
每次重新打开 notebook **先跑本 cell**。

In [1]:
import importlib
import os
import sys

from IPython.display import display


def _bootstrap_src() -> str:
    cur = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(cur, "任务.txt")):
            src = os.path.join(cur, "src")
            if src not in sys.path:
                sys.path.insert(0, src)
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        "未能定位工程根。请在 VS Code 用 Open Folder 打开「02 数据处理」。"
    )


PROJECT_DIR = _bootstrap_src()
os.environ["PROJECT_DIR"] = PROJECT_DIR

import load_pipeline as _lp

importlib.reload(_lp)
from load_pipeline import TOKENIZER_MODEL_ID, setup_paths

PATHS = setup_paths(PROJECT_DIR)
TABLES_DIR = PATHS["tables_dir"]

print(f"PROJECT_DIR : {PROJECT_DIR}")
print(f"TABLES_DIR  : {TABLES_DIR}  （partA 验证期对照表）")
print(f"Tokenizer   : {TOKENIZER_MODEL_ID}")
print("\n[OK] 前置 — 可继续 B0")

c:\Users\10138\miniconda3\envs\med-rag-verify\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_DIR : d:\谷歌\02 数据处理
TABLES_DIR  : d:\谷歌\02 数据处理\outputs\tables  （partA 验证期对照表）
Tokenizer   : sentence-transformers/all-MiniLM-L6-v2

[OK] 前置 — 可继续 B0


---

## 流程总览

| 步骤 | 说明 |
|------|------|
| **B0** | 外接盘挂载、`MED_RAG_DATA_ROOT` |
| **B1** | 下载 `deprecated/oa_bulk/oa_comm/xml/*.tar.gz` → `raw/` |
| **B2** | 解压 → `extracted/`，校验后删 `.tar.gz` |
| **B3** | `pmcid_index` + `oa_comm_slim.jsonl` |
| **B4** | 全量行数、丢弃率统计 |
| **B5** | 抽样 vs partA（97 篇）策略对比 |
| **B6** | 结论与后续 RAG 建议 |

试点建议：`PILOT_MAX_TARS=2`，`DOWNLOAD_BASELINE_ONLY=True`（13 个 baseline 包）。

## 【B0】外接盘环境与路径检查

插入硬盘后运行。若盘符不同，修改 `MED_RAG_DATA_ROOT`。

In [2]:
import importlib
import json
import os

import full_scale_pipeline as _fsp

importlib.reload(_fsp)
from full_scale_pipeline import (
    FullScalePaths,
    quick_check_data_root,
    list_pmc_subdirs,
    count_tar_gz,
)

# 跨平台默认路径：Windows 用 E: 盘，Mac 用 /Volumes/Lexar
_DEFAULT_ROOT = "E:/med-llm-rag-datasets" if os.name == "nt" else "/Volumes/Lexar/med-llm-rag-datasets"
MED_RAG_DATA_ROOT = os.environ.get("MED_RAG_DATA_ROOT", _DEFAULT_ROOT)
FS = FullScalePaths.from_env(MED_RAG_DATA_ROOT)
FS.apply_env()

# 快速检查（不统计XML文件数量）
mount = quick_check_data_root(FS)
print(json.dumps(mount, indent=2, ensure_ascii=False))
if not mount["mounted"]:
    raise RuntimeError(f"外接盘未挂载: {FS.data_root}")
if not mount["writable"]:
    raise RuntimeError(f"外接盘不可写: {FS.data_root}")

# 快速显示子目录（不递归统计）
subdirs = list_pmc_subdirs(FS.extracted_dir)
print(f"\nraw/       : {FS.raw_dir}  (tar.gz: {count_tar_gz(FS.raw_dir)})")
print(f"extracted/ : {FS.extracted_dir}  ({len(subdirs)} 个PMC子文件夹)")
print(f"slim jsonl : {FS.slim_jsonl}")
print("\n[OK] B0 — 路径就绪（如需统计XML文件数量，运行下方B0-stat单元格）")

{
  "data_root": "E:\\med-llm-rag-datasets",
  "mounted": true,
  "writable": true,
  "disk_total_gb": 5000.9,
  "disk_free_gb": 987.7,
  "raw_exists": true,
  "extracted_exists": true,
  "processed_exists": true
}

raw/       : E:\med-llm-rag-datasets\raw  (tar.gz: 13)
extracted/ : E:\med-llm-rag-datasets\extracted  (13 个PMC子文件夹)
slim jsonl : E:\med-llm-rag-datasets\processed\oa_comm_slim.jsonl

[OK] B0 — 路径就绪（如需统计XML文件数量，运行下方B0-stat单元格）


## 【B0-stat】（可选）统计XML文件数量

大目录统计耗时较长（约500万文件需要几分钟）。  
**仅在需要确认XML总数时运行**，日常启动notebook无需执行。

In [ ]:
from full_scale_pipeline import count_xml_files

print("正在统计XML文件数量（大目录可能需要几分钟）...")
xml_count = count_xml_files(FS.extracted_dir, use_cache=False)  # 强制重新统计
print(f"extracted/ XML文件总数: {xml_count:,}")

## 【B1】下载 PMC oa_comm 原始包

- FTP 目录：`https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_bulk/oa_comm/xml/`
- 落盘：`$MED_RAG_DATA_ROOT/raw/*.tar.gz`

| 参数 | 含义 |
|------|------|
| `SKIP_DOWNLOAD` | 已下载则跳过 |
| `DOWNLOAD_BASELINE_ONLY` | `True` = 仅 13 个 baseline；`False` = 含 incr（~129） |
| `PILOT_MAX_TARS` | 本次最多下几个；`None` = 不限制 |
| `FORCE_REFRESH_FILE_LIST` | 清单 404 后改 `True` 一次；成功后再改 `False` |

In [ ]:
import importlib
import full_scale_pipeline as _fsp

importlib.reload(_fsp)
from full_scale_pipeline import download_archives, fetch_oa_comm_file_list

SKIP_DOWNLOAD = True    # ← 改为 True（已手动下载）
PILOT_MAX_TARS = 2      # 这行无所谓了，会被跳过
DOWNLOAD_BASELINE_ONLY = True
FORCE_REFRESH_FILE_LIST = True

FILE_LIST_CACHE = os.path.join(FS.stats_dir, "oa_comm_file_list.txt")

if SKIP_DOWNLOAD:
    print("[跳过] B1 — 使用 raw/ 已有 tar.gz")
else:
    names = fetch_oa_comm_file_list(
        cache_path=FILE_LIST_CACHE,
        force_refresh=FORCE_REFRESH_FILE_LIST,
        baseline_only=DOWNLOAD_BASELINE_ONLY,
        include_incr=not DOWNLOAD_BASELINE_ONLY,
    )
    print(f"清单共 {len(names)} 个 tar.gz；本次上限: {PILOT_MAX_TARS or '全部'}")
    dl_results = download_archives(
        names, FS.raw_dir, max_files=PILOT_MAX_TARS, skip_existing=True
    )
    ok = sum(1 for r in dl_results if r.get("status") in ("ok", "skipped"))
    err = [r for r in dl_results if r.get("status") == "error"]
    print(f"[OK] B1 — 完成 {ok} 个，失败 {len(err)} 个")
    if err:
        display(err[:5])
    dl_log = os.path.join(FS.stats_dir, "download_log.json")
    with open(dl_log, "w", encoding="utf-8") as f:
        json.dump(dl_results, f, ensure_ascii=False, indent=2)
    print(f"日志: {dl_log}")

print(f"raw/ tar.gz 数量: {count_tar_gz(FS.raw_dir)}")

## 【B2】解压、校验并删除压缩包

解压到 `extracted/`。校验通过后删除 `.tar.gz`（`DELETE_ARCHIVES_AFTER_VERIFY=False` 可保留备份）。

In [ ]:
import json
from pathlib import Path

import full_scale_pipeline as _fsp
importlib.reload(_fsp)
from full_scale_pipeline import extract_all_archives, remove_archives

SKIP_EXTRACT = False
DELETE_ARCHIVES_AFTER_VERIFY = False   # 解压后不删除 tar.gz
DRY_RUN_EXTRACT = False
SKIP_EXISTING = True  # 启用断点续传：跳过已解压的文件

if SKIP_EXTRACT:
    print("[跳过] B2 — 使用已有 extracted/")
else:
    reports = extract_all_archives(
        FS.raw_dir, FS.extracted_dir, 
        dry_run=DRY_RUN_EXTRACT,
        skip_existing=SKIP_EXISTING  # 断点续传：检测已解压文件并跳过
    )
    ok_names = {r.archive for r in reports if r.extracted_ok}
    fail = [r for r in reports if not r.extracted_ok]
    print(f"解压: {len(reports)} 包 | 成功 {len(ok_names)} | 失败 {len(fail)}")
    for r in fail[:5]:
        print(f"  FAIL {r.archive}: {r.error}")

    extract_log = os.path.join(FS.stats_dir, "extract_report.json")
    with open(extract_log, "w", encoding="utf-8") as f:
        json.dump([r.to_dict() for r in reports], f, ensure_ascii=False, indent=2)

    if DELETE_ARCHIVES_AFTER_VERIFY and not DRY_RUN_EXTRACT:
        archives = [str(p) for p in Path(FS.raw_dir).glob("*.tar.gz")]
        removed = remove_archives(archives, only_if_ok=True, ok_names=ok_names)
        print(f"已删除压缩包: {len(removed)} 个")

print(f"extracted/ XML: {count_xml_files(FS.extracted_dir)}")
print("[OK] B2")

## 【B3】构建 slim JSONL（分批处理，支持断点续传）

按13个PMC子文件夹分批构建jsonl，每个子文件夹生成一个shard文件。  
支持中断后继续：已完成的子文件夹会自动跳过。

| 参数 | 说明 |
|------|------|
| `RESUME` | `True` = 断点续传（跳过已完成的子文件夹） |
| `MERGE_SHARDS` | `True` = 最后合并所有shard为一个jsonl |

In [4]:
import json
import os

import full_scale_pipeline as _fsp
importlib.reload(_fsp)
from full_scale_pipeline import (
    build_jsonl_batched,
    merge_jsonl_shards,
    get_batch_progress,
    list_pmc_subdirs,
)

# 配置参数
RESUME = True           # 断点续传：跳过已完成的子文件夹
MERGE_SHARDS = True     # 完成后合并所有shard文件
SKIP_BUILD = False      # 完全跳过构建（使用已有结果）

# 检查是否有XML文件
subdirs = list_pmc_subdirs(FS.extracted_dir)
if not subdirs:
    raise RuntimeError("extracted/ 无PMC子文件夹，请先完成 B2 解压")

print(f"发现 {len(subdirs)} 个PMC子文件夹:")
for sd in subdirs:
    print(f"  - {sd['name']}")

# 查看当前进度
progress = get_batch_progress(FS.processed_dir)
print(f"\n已完成: {len(progress['completed'])}/{len(subdirs)} 个子文件夹")

if SKIP_BUILD:
    print("\n[跳过] B3 — 使用已有结果")
else:
    print("\n开始分批构建...")
    progress = build_jsonl_batched(
        FS.extracted_dir,
        FS.processed_dir,
        slim=True,
        skip_no_abstract=True,
        resume=RESUME,
    )
    
    print(f"\n分批构建完成:")
    print(f"  已处理子文件夹: {len(progress['completed'])}")
    print(f"  总记录数: {progress.get('total_ok', 0):,}")
    
    # 合并shard文件
    if MERGE_SHARDS and len(progress['completed']) == len(subdirs):
        print(f"\n合并shard文件到 {FS.slim_jsonl} ...")
        total = merge_jsonl_shards(FS.processed_dir, FS.slim_jsonl)
        print(f"合并完成: {total:,} 行")
    elif not MERGE_SHARDS:
        print("\n[提示] 未合并shard文件，可单独运行合并步骤")
    else:
        print(f"\n[提示] 尚有 {len(subdirs) - len(progress['completed'])} 个子文件夹未完成，暂不合并")

print("\n[OK] B3")

发现 13 个PMC子文件夹:
  - PMC000xxxxxx
  - PMC001xxxxxx
  - PMC002xxxxxx
  - PMC003xxxxxx
  - PMC004xxxxxx
  - PMC005xxxxxx
  - PMC006xxxxxx
  - PMC007xxxxxx
  - PMC008xxxxxx
  - PMC009xxxxxx
  - PMC010xxxxxx
  - PMC011xxxxxx
  - PMC012xxxxxx

已完成: 12/13 个子文件夹

开始分批构建...
共 13 个子文件夹，已完成 12 个
[1/13] PMC000xxxxxx — 已完成，跳过
[2/13] PMC001xxxxxx — 已完成，跳过
[3/13] PMC002xxxxxx — 已完成，跳过
[4/13] PMC003xxxxxx — 已完成，跳过
[5/13] PMC004xxxxxx — 已完成，跳过
[6/13] PMC005xxxxxx — 已完成，跳过
[7/13] PMC006xxxxxx — 已完成，跳过
[8/13] PMC007xxxxxx — 已完成，跳过
[9/13] PMC008xxxxxx — 已完成，跳过
[10/13] 处理 PMC009xxxxxx ...


PMC009xxxxxx: 100%|██████████| 604303/604303 [6:45:16<00:00, 24.85it/s]     


    完成: ok=532208, dropped=72094, failed=1
[11/13] PMC010xxxxxx — 已完成，跳过
[12/13] PMC011xxxxxx — 已完成，跳过
[13/13] PMC012xxxxxx — 已完成，跳过

分批构建完成:
  已处理子文件夹: 13
  总记录数: 4,557,627

合并shard文件到 E:\med-llm-rag-datasets\processed\oa_comm_slim.jsonl ...
合并 PMC000xxxxxx.jsonl ...
合并 PMC001xxxxxx.jsonl ...
合并 PMC002xxxxxx.jsonl ...
合并 PMC003xxxxxx.jsonl ...
合并 PMC004xxxxxx.jsonl ...
合并 PMC005xxxxxx.jsonl ...
合并 PMC006xxxxxx.jsonl ...
合并 PMC007xxxxxx.jsonl ...
合并 PMC008xxxxxx.jsonl ...
合并 PMC009xxxxxx.jsonl ...
合并 PMC010xxxxxx.jsonl ...
合并 PMC011xxxxxx.jsonl ...
合并 PMC012xxxxxx.jsonl ...
合并完成: 4557627 行 → E:\med-llm-rag-datasets\processed\oa_comm_slim.jsonl
合并完成: 4,557,627 行

[OK] B3


发现 13 个PMC子文件夹:
  - PMC000xxxxxx
  - PMC001xxxxxx
  - PMC002xxxxxx
  - PMC003xxxxxx
  - PMC004xxxxxx
  - PMC005xxxxxx
  - PMC006xxxxxx
  - PMC007xxxxxx
  - PMC008xxxxxx
  - PMC009xxxxxx
  - PMC010xxxxxx
  - PMC011xxxxxx
  - PMC012xxxxxx

已完成: 0/13 个子文件夹

开始分批构建...
共 13 个子文件夹，已完成 0 个
[1/13] 处理 PMC000xxxxxx ...
PMC000xxxxxx: 100%|██████████| 3028/3028 [00:56<00:00, 53.83it/s]
    完成: ok=2775, dropped=253, failed=0
[2/13] 处理 PMC001xxxxxx ...
PMC001xxxxxx: 100%|██████████| 27518/27518 [13:36<00:00, 33.71it/s] 
    完成: ok=24466, dropped=3052, failed=0
[3/13] 处理 PMC002xxxxxx ...
PMC002xxxxxx: 100%|██████████| 122578/122578 [55:04<00:00, 37.09it/s]  
    完成: ok=111250, dropped=11327, failed=1
[4/13] 处理 PMC003xxxxxx ...
PMC003xxxxxx: 100%|██████████| 323744/323744 [3:09:43<00:00, 28.44it/s]   
    完成: ok=293323, dropped=30419, failed=0
[5/13] 处理 PMC004xxxxxx ...
PMC004xxxxxx: 100%|██████████| 392737/392737 [3:00:59<00:00, 36.17it/s]    
    完成: ok=357255, dropped=35480, failed=2
[6/13] 处理 PMC005xxxxxx ...
PMC005xxxxxx: 100%|██████████| 435897/435897 [6:15:20<00:00, 19.36it/s]    
    完成: ok=362708, dropped=73159, failed=2
[7/13] 处理 PMC006xxxxxx ...
PMC006xxxxxx: 100%|██████████| 462078/462078 [4:51:06<00:00, 26.46it/s]    
    完成: ok=426070, dropped=36005, failed=3
[8/13] 处理 PMC007xxxxxx ...
PMC007xxxxxx: 100%|██████████| 465525/465525 [2:09:47<00:00, 59.78it/s]   
    完成: ok=443293, dropped=22231, failed=1
[9/13] 处理 PMC008xxxxxx ...
PMC008xxxxxx: 100%|██████████| 550502/550502 [6:37:37<00:00, 23.07it/s]    
    完成: ok=508703, dropped=41795, failed=4
[10/13] 处理 PMC009xxxxxx ...
PMC009xxxxxx: 0it [00:00, ?it/s]
    完成: ok=0, dropped=0, failed=0
[11/13] 处理 PMC010xxxxxx ...
PMC010xxxxxx: 100%|██████████| 610830/610830 [10:02:56<00:00, 16.88it/s]   
    完成: ok=553423, dropped=57407, failed=0
[12/13] 处理 PMC011xxxxxx ...
PMC011xxxxxx: 100%|██████████| 547391/547391 [5:10:05<00:00, 29.42it/s]    
    完成: ok=517838, dropped=29551, failed=2
[13/13] 处理 PMC012xxxxxx ...
PMC012xxxxxx: 100%|██████████| 448065/448065 [5:51:49<00:00, 21.23it/s]    
    完成: ok=424315, dropped=23748, failed=2

分批构建完成:
  已处理子文件夹: 13
  总记录数: 4,025,419

合并shard文件到 E:\med-llm-rag-datasets\processed\oa_comm_slim.jsonl ...
合并 PMC000xxxxxx.jsonl ...
合并 PMC001xxxxxx.jsonl ...
合并 PMC002xxxxxx.jsonl ...
合并 PMC003xxxxxx.jsonl ...
合并 PMC004xxxxxx.jsonl ...
合并 PMC005xxxxxx.jsonl ...
合并 PMC006xxxxxx.jsonl ...
合并 PMC007xxxxxx.jsonl ...
合并 PMC008xxxxxx.jsonl ...
合并 PMC009xxxxxx.jsonl ...
合并 PMC010xxxxxx.jsonl ...
合并 PMC011xxxxxx.jsonl ...
合并 PMC012xxxxxx.jsonl ...
合并完成: 4025419 行 → E:\med-llm-rag-datasets\processed\oa_comm_slim.jsonl
合并完成: 4,025,419 行

[OK] B3

## 【B4】全量 slim 规模与清洗统计

In [5]:
import os

import pandas as pd
from full_scale_pipeline import count_jsonl_lines, count_skipped, get_batch_progress

# 从进度文件获取统计（避免重新扫描500万文件）
progress = get_batch_progress(FS.processed_dir)
n_slim = count_jsonl_lines(FS.slim_jsonl) if os.path.isfile(FS.slim_jsonl) else progress.get('total_ok', 0)
n_skip = count_skipped(FS.skipped_log)
denom = n_slim + n_skip
skip_rate = round(100 * n_skip / denom, 2) if denom else 0.0

# 从B3进度获取处理的XML数量（近似值，避免重新统计）
total_stats = progress.get('stats', {})
estimated_xml = sum(s.get('ok', 0) + s.get('dropped_no_abstract', 0) + s.get('failed', 0) 
                    for s in total_stats.values())

full_build_stats = pd.DataFrame(
    [
        {"item": "处理的XML数（估算）", "value": estimated_xml},
        {"item": "slim jsonl 行数", "value": n_slim},
        {"item": "skipped_no_abstract", "value": n_skip},
        {"item": "abstract 丢弃率 (%)", "value": skip_rate},
        {
            "item": "slim 文件大小 (MB)",
            "value": round(os.path.getsize(FS.slim_jsonl) / 1e6, 1) if os.path.isfile(FS.slim_jsonl) else 0,
        },
    ]
)
display(full_build_stats)
stats_path = os.path.join(FS.stats_dir, "full_build_stats.csv")
full_build_stats.to_csv(stats_path, index=False)
print(f"已保存: {stats_path}")
print("[OK] B4")

,item,value
0,处理的XML数（估算）,4994166.00
1,slim jsonl 行数,4557627.00
2,skipped_no_abstract,436521.00
3,abstract 丢弃率 (%),8.74
4,slim 文件大小 (MB),8896.60


已保存: E:\med-llm-rag-datasets\processed\stats\full_build_stats.csv
[OK] B4


## 【B5】策略抽样验证（vs partA 验证期 97 篇）

从全量 slim 随机抽样（默认 5000 篇），对比 `outputs/tables/token_percentiles.csv` 等。

In [6]:
import importlib
import os

import pandas as pd
import chunk_strategy as _cs
from chunk_strategy import ChunkStrategyConfig, summarize_retrieval_chunks
from domain_analysis import load_tokenizer
from full_scale_pipeline import (
    compare_to_baseline,
    count_jsonl_lines,
    count_skipped,
    load_verify_baseline,
    read_jsonl_sample,
)
from token_stats import add_all_token_lens, token_percentile_table

importlib.reload(_cs)

VALIDATION_SAMPLE_N = 5000
VALIDATION_SEED = 42

n_slim = count_jsonl_lines(FS.slim_jsonl)
n_skip = count_skipped(FS.skipped_log)
skip_rate = round(100 * n_skip / (n_slim + n_skip), 2) if (n_slim + n_skip) else 0.0

baseline = load_verify_baseline(TABLES_DIR)
print("partA 对照:", baseline)

rows = read_jsonl_sample(
    FS.slim_jsonl, min(VALIDATION_SAMPLE_N, n_slim), seed=VALIDATION_SEED
)
df_full_sample = pd.DataFrame(rows)
print(f"抽样 {len(df_full_sample)} 篇 / 全量 {n_slim} 篇")

tokenizer = load_tokenizer(TOKENIZER_MODEL_ID)
df_tok = add_all_token_lens(df_full_sample, tokenizer)
pct_full = token_percentile_table(df_tok)
pct_path = os.path.join(FS.stats_dir, "token_percentiles_full_sample.csv")
pct_full.to_csv(pct_path, index=False)
display(pct_full)

pct_over_512 = round(100 * (df_tok["retrieval_token_len"] > 512).mean(), 1)
chunk_full = summarize_retrieval_chunks(
    df_full_sample, tokenizer, config=ChunkStrategyConfig()
)
sub = chunk_full[chunk_full["pmcid"] != "__summary__"]
single_pct = round(100 * (sub["n_chunks"] == 1).mean(), 1)
multi_pct = round(100 * (sub["n_chunks"] > 1).mean(), 1)
row_ret = pct_full[pct_full["field"] == "title+abstract"]
p95_full = float(row_ret["p95"].iloc[0]) if not row_ret.empty else None

full_metrics = {
    "abstract_skip_rate_pct": skip_rate,
    "p95_retrieval": p95_full,
    "pct_over_512": pct_over_512,
    "single_pct": single_pct,
    "multi_pct": multi_pct,
}
cmp = compare_to_baseline(full_metrics, baseline)
cmp_path = os.path.join(FS.stats_dir, "verify_vs_full_compare.csv")
cmp.to_csv(cmp_path, index=False)
display(cmp)

chunk_path = os.path.join(FS.stats_dir, "chunk_strategy_summary_full_sample.csv")
chunk_full.to_csv(chunk_path, index=False)
print(f"\n{pct_path}\n{cmp_path}")
print("[OK] B5")

partA 对照: {'n_clean': 97, 'n_sample': 100, 'p95_retrieval': 617.2, 'p99_retrieval': 1009.0, 'single_pct': np.float64(85.6), 'multi_pct': np.float64(14.4), 'pct_over_512': 14.4}
抽样 5000 篇 / 全量 4557627 篇


c:\Users\10138\miniconda3\envs\med-rag-verify\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\10138\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


,field,n,mean,min,p50,p75,p95,p99,max
0,title,5000,23.65,2.0,23.0,29.0,40.0,51.0,78.0
1,abstract,5000,352.67,6.0,340.0,427.0,584.0,766.0,1902.0
2,title+abstract,5000,376.32,29.0,365.0,453.0,612.0,793.0,1919.0
3,body,5000,0.00,0.0,0.0,0.0,0.0,0.0,0.0


,metric,verify_97,full_sample,delta,within_tolerance
0,abstract 丢弃率 (%),3.0,8.74,5.74,False
1,P95 retrieval tokens,617.2,612.00,-5.20,True
2,>512 占比 (%),14.4,13.70,-0.70,True
3,单块占比 (%),85.6,86.40,0.80,True
4,多块占比 (%),14.4,13.60,-0.80,True



E:\med-llm-rag-datasets\processed\stats\token_percentiles_full_sample.csv
E:\med-llm-rag-datasets\processed\stats\verify_vs_full_compare.csv
[OK] B5


## 【B6】全量验证结论

写入外接盘 `processed/stats/full_scale_verdict.txt`，并更新《RAG数据分析与设计说明》§6.4。

In [7]:
import os

from full_scale_pipeline import strategy_verdict

verdict = strategy_verdict(cmp)
print("=" * 60)
print(verdict)
print("=" * 60)

cfg_path = os.path.join(TABLES_DIR, "chunk_strategy_config.json")
if os.path.isfile(cfg_path):
    print("\npartA 定稿参数:")
    print(open(cfg_path, encoding="utf-8").read())

verdict_path = os.path.join(FS.stats_dir, "full_scale_verdict.txt")
with open(verdict_path, "w", encoding="utf-8") as f:
    f.write(verdict + "\n")
print(f"\n已保存: {verdict_path}")
print("[OK] B6 — 阶段 B 完成")

以下指标偏离验证期较多：abstract 丢弃率 (%)。建议先复核 slim 构建与抽样规模，再决定是否微调切块参数或清洗规则；并更新《RAG数据分析与设计说明》「全量 vs 验证期」小节。

partA 定稿参数:
{
  "retrieval_unit": "title+abstract",
  "retrieval_token_limit": 512,
  "retrieval_chunk_size": 400,
  "retrieval_chunk_overlap": 80,
  "body_chunk_size": 512,
  "body_chunk_overlap": 80,
  "drop_missing_abstract": true,
  "drop_short_abstract": false,
  "short_abstract_char_threshold": 50,
  "notes": [
    "主体 ~85% title+abstract ≤512：单 chunk",
    "长尾 ~14% >512：RecursiveCharacterTextSplitter",
    "body 一律 sliding_window，与摘要分开",
    "无 abstract：丢弃并记录 pmcid"
  ]
}

已保存: E:\med-llm-rag-datasets\processed\stats\full_scale_verdict.txt
[OK] B6 — 阶段 B 完成
